In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving diabetes.csv to diabetes.csv


In [ ]:
df = pd.read_csv("diabetes.csv")

In [ ]:
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72.0,35,169.5,33.6,0.627,50,1
1,1,85,66.0,29,102.5,26.6,0.351,31,0
2,8,183,64.0,32,169.5,23.3,0.672,32,1
3,1,89,66.0,23,94.0,28.1,0.167,21,0
4,0,137,40.0,35,168.0,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76.0,48,180.0,32.9,0.171,63,0
764,2,122,70.0,27,102.5,36.8,0.340,27,0
765,5,121,72.0,23,112.0,26.2,0.245,30,0
766,1,126,60.0,32,169.5,30.1,0.349,47,1


In [ ]:
x = df.iloc[:,:-1]
y= df.iloc[:,-1]

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit_transform(x)

array([[ 0.63994726,  0.86462486, -0.03218035, ...,  0.16948251,
         0.46849198,  1.4259954 ],
       [-0.84488505, -1.20472661, -0.52812374, ..., -0.84854874,
        -0.36506078, -0.19067191],
       [ 1.23388019,  2.01426457, -0.69343821, ..., -1.32847775,
         0.60439732, -0.10558415],
       ...,
       [ 0.3429808 , -0.02224005, -0.03218035, ..., -0.90672195,
        -0.68519336, -0.27575966],
       [-0.84488505,  0.14199419, -1.02406713, ..., -0.33953311,
        -0.37110101,  1.17073215],
       [-0.84488505, -0.94195182, -0.19749482, ..., -0.2959032 ,
        -0.47378505, -0.87137393]])

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,random_state=2,test_size=0.2)

In [ ]:
import tensorflow
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense,Input

In [ ]:
!pip install keras-tuner -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 6.5 MB/s eta 0:00:00


In [ ]:
import keras_tuner as kt

In [ ]:
def build_model(hp):
    model = Sequential()

    # Input layer
    model.add(Input(shape=(8,)))

    num_layers = hp.Int(
        "num_layers",
        min_value=1,
        max_value=5
    )

    units = hp.Int(
        "units",
        min_value=8,
        max_value=128,
        step=8
    )

    activation = hp.Choice(
        "activation",
        values=["relu", "tanh", "sigmoid"]
    )

    # Hidden layers
    for i in range(num_layers):
        model.add(Dense(units, activation=activation))

    # Output layer
    model.add(Dense(1, activation="sigmoid"))

    optimizer = hp.Choice(
        "optimizer",
        values=["adam", "rmsprop", "sgd","adadelta"]
    )

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
tuner=kt.RandomSearch(build_model,max_trials=5,objective="val_accuracy")

Reloading Tuner from ./untitled_project/tuner0.json


In [ ]:
tuner.search(x_train,y_train,epochs=7,validation_data=(x_test,y_test))


In [ ]:
tuner.get_best_hyperparameters(num_trials=1)[0].values

{'num_layers': 2, 'units': 128, 'activation': 'relu', 'optimizer': 'sgd'}

In [ ]:
model = tuner.get_best_models(num_models=1)[0]

In [ ]:
model.fit(x_train,y_train,epochs=100,initial_epoch=8,validation_data=(x_test,y_test))

Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8371 - loss: 0.3788 - val_accuracy: 0.7792 - val_loss: 0.4726
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8322 - loss: 0.3895 - val_accuracy: 0.7857 - val_loss: 0.4699
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8241 - loss: 0.3982 - val_accuracy: 0.7792 - val_loss: 0.5076
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8208 - loss: 0.4101 - val_accuracy: 0.7987 - val_loss: 0.4922
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8322 - loss: 0.3916 - val_accuracy: 0.7792 - val_loss: 0.5638
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8225 - loss: 0.4007 - val_accuracy: 0.7597 - val_loss: 0.5265
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8274 - loss: 0.3898 - val_accuracy: 0.7662 - val_loss: 0.4903
Epoch 15/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8241 - loss: 0.4025 - val_accuracy: 0.79

See we have observed that after using the hyperparameter fine-tuning we can easily get the highest val accuracy, which reduces the overfitting on the data.